### 토큰 비용 계산기
- 파일 : nA4_cost_calc.ipynb
- 역할 : 클라우드와 온디바이스의 월 비용과 손익분기를 계산하는 계산기를 만듭니다.
- 기능 :
    1. 클라우드 월 비용 함수  
    2. 온디바이스 월 비용 함수  
    3. 손익분기와 용량 확인  
    4. 사용량-비용 그래프  


# 4장 클라우드와 온디바이스 비용 비교: 토큰 비용 계산기

부록 원고의 코드와 동일합니다. 핵심 수식은 코드의 주석에 함께 적어 두었습니다.

>> [코드 A4-1] 클라우드 월 비용 함수 (nA4_cost_calc.ipynb, STEP 1)
<hr>

In [ ]:
# 가격은 2026년 9월 기준 예시이며, 실행 시점의 공식 가격으로 바꿔 넣습니다

#- 요금표 (실행 시점의 공식 가격으로 교체)
PRICE = {
    "mid":   {"in": 2.00, "out": 12.00},   # 중간급 모델 (달러/1M 토큰)
    "light": {"in": 0.20, "out": 1.20},    # 경량 모델
}

#- 클라우드 월 비용 함수
def cloud_monthly(req_per_day, tok_in, tok_out, price, days=30):
    m_in  = req_per_day * days * tok_in  / 1e6   # 월 입력 토큰 (M)
    m_out = req_per_day * days * tok_out / 1e6   # 월 출력 토큰 (M)
    # 월 비용 = 입력(M) × 입력 단가 + 출력(M) × 출력 단가
    return m_in * price["in"] + m_out * price["out"]

#- 하루 500건 시나리오로 확인
for name in PRICE:
    cost = cloud_monthly(500, 400, 300, PRICE[name])
    print(f"{name:5s} 모델: 월 {cost:.1f}달러")

# 확인 포인트 -------------------------------------------------------------------------
# - mid 66.0달러, light 6.6달러 출력 체크

**실행 결과 예시**
```
mid   모델: 월 66.0달러
light 모델: 월 6.6달러
```

>> [코드 A4-2] 온디바이스 월 비용 함수 (STEP 2)
<hr>

In [ ]:
#- 온디바이스 월 비용: 상각 + 전력
def ondevice_monthly(device_usd, life_months, watts,
                     krw_per_kwh=160, krw_per_usd=1400, days=30):
    # 전기요금과 환율은 예시 입력이며 실제 조건으로 바꿔 넣습니다
    amort = device_usd / life_months                     # 월 상각액
    kwh   = watts * 24 * days / 1000                     # 월 전력량
    elec  = kwh * krw_per_kwh / krw_per_usd              # 월 전력비 (달러)
    return amort + elec, amort, elec

#- AI HAT+ 2와 Pi 5 출시가 합계 210달러(2026년 1월 기준, 구매 시점 가격으로 교체), 36개월, 1.87W
total, amort, elec = ondevice_monthly(device_usd=210, life_months=36, watts=1.87)
print(f"상각 {amort:.2f}달러 + 전력 {elec:.2f}달러 = 월 {total:.2f}달러")

# 확인 포인트 --------------------------------------------------------------------------
# - 전력비의 상각액 대비 비율(%) 계산

**실행 결과 예시**
```
상각 5.83달러 + 전력 0.15달러 = 월 5.99달러
```

>> [코드 A4-3] 손익분기와 용량 확인 (STEP 3)
<hr>

In [ ]:
#- 시나리오 사용량
tok_in, tok_out = 400, 300
price = PRICE["mid"]

# 요청 1,000건당 클라우드 비용에서 가중 평균 단가(달러/1M 토큰)를 구합니다

#- 가중 평균 단가와 손익분기
mix = (tok_in * price["in"] + tok_out * price["out"]) / (tok_in + tok_out)

# 손익분기 월 토큰(M) = 온디바이스 월 고정비(total) / 가중 평균 단가(mix)
be_mtok = total / mix
be_req_day = be_mtok * 1e6 / (tok_in + tok_out) / 30
print(f"가중 평균 단가 {mix:.2f}달러/M, 손익분기 월 {be_mtok:.2f}M 토큰 (하루 {be_req_day:.0f}건)")

#- 장비 1대의 월 출력 상한과 필요량
cap_mtok = 6.9 * 86400 * 30 / 1e6   # 초당 6.9토큰 기준 월 출력 상한
need_mtok = 500 * 30 * tok_out / 1e6
print(f"장비 1대 월 출력 상한 {cap_mtok:.1f}M, 시나리오 필요량 {need_mtok:.1f}M")

# 확인 포인트 -------------------------------------------------------------------------
# - 하루 500건의 손익분기 대비 배수 계산, 용량 이내 체크

**실행 결과 예시**
```
가중 평균 단가 6.29달러/M, 손익분기 월 0.95M 토큰 (하루 45건)
장비 1대 월 출력 상한 17.9M, 시나리오 필요량 4.5M
```

>> [코드 A4-4] 사용량-비용 그래프 (STEP 4)
<hr>

In [ ]:
#- 모듈로딩
import numpy as np
import matplotlib.pyplot as plt

#- 하루 0~1,000건 구간의 월 비용 곡선
reqs = np.arange(0, 1001, 10)          # 하루 요청 수 0~1,000건
cloud_mid   = [cloud_monthly(r, tok_in, tok_out, PRICE["mid"])   for r in reqs]
cloud_light = [cloud_monthly(r, tok_in, tok_out, PRICE["light"]) for r in reqs]

# 온디바이스는 사용량과 무관한 수평선
ondev = [total] * len(reqs)

#- 세 선을 한 그래프에
plt.figure(figsize=(7, 4))
plt.plot(reqs, cloud_mid, label="Cloud (mid)")
plt.plot(reqs, cloud_light, label="Cloud (light)")
plt.plot(reqs, ondev, label="On-device")
plt.xlabel("requests / day")
plt.ylabel("USD / month")
plt.legend()
plt.tight_layout()
plt.show()

# 확인 포인트 ------------------------------------------
# - 중간급 모델 선과 온디바이스 선의 교차점(하루 건수) 확인
# - 경량 모델 선과 온디바이스 선의 관계를 앞의 민감도 분석과 연결해 정리
# - PRICE를 최신 공식 가격으로 바꿀 때 교차점 이동 확인